# 🚀 TurboScript Backend — Free GPU Transcription Server

This notebook runs a **Whisper transcription API** on Google Colab's free T4 GPU.

### Quick Start
1. **Runtime → Change runtime type → T4 GPU** (free)
2. Run all cells (**Runtime → Run all**)
3. Copy the **ngrok URL** printed at the end
4. Paste it into **TurboScript Settings → Colab URL**

> 💡 The server stays alive as long as the Colab tab is open. Free sessions last up to 12 hours.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q faster-whisper fastapi uvicorn pyngrok python-multipart nest-asyncio
print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Check GPU ─────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f"🎮 GPU detected: {result.stdout.strip()}")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
    print("    CPU mode will work but is MUCH slower.")

In [ ]:
# ── Cell 3: Load Whisper model ────────────────────────────────────────────────
# Model options:
#   tiny, base, small, medium   → fast, lower accuracy
#   large-v2, large-v3          → best accuracy (recommended on T4)
#   distil-large-v3             → fast + accurate (great balance)

from faster_whisper import WhisperModel
import torch

MODEL_SIZE = "large-v3"   # ← Change this if you want a different model

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"⏳ Loading {MODEL_SIZE} on {device} ({compute_type})...")
model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)
print(f"✅ Model loaded: whisper-{MODEL_SIZE} on {device.upper()}")

In [ ]:
# ── Cell 4: Create FastAPI server ─────────────────────────────────────────────
import os
import tempfile
import threading
import uvicorn
import nest_asyncio
from fastapi import FastAPI, UploadFile, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from typing import Optional

nest_asyncio.apply()

app = FastAPI(
    title="TurboScript Transcription API",
    description="Whisper transcription server running on Google Colab GPU",
    version="1.0.0",
)

# Allow all origins (needed for web app CORS)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
async def health():
    """Health check endpoint — used by the web app to verify the connection."""
    return {
        "status": "ok",
        "model": MODEL_SIZE,
        "device": device,
        "compute_type": compute_type,
    }


@app.post("/transcribe")
async def transcribe(
    file: UploadFile,
    language: Optional[str] = Form(None),
    task: Optional[str] = Form("transcribe"),
):
    """
    Transcribe an audio/video file.
    
    Parameters:
    - file: Audio or video file (mp3, mp4, wav, m4a, ogg, webm, flac, etc.)
    - language: ISO language code (e.g. 'en', 'es', 'fr'). None = auto-detect.
    - task: 'transcribe' or 'translate' (translate → English output)
    """
    # Determine file extension
    filename = file.filename or "audio.mp3"
    ext = "." + filename.rsplit(".", 1)[-1].lower() if "." in filename else ".mp3"
    
    # Write to temp file
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        content = await file.read()
        tmp.write(content)
        tmp_path = tmp.name
    
    try:
        # Transcribe with faster-whisper
        lang = language if language and language not in ("", "auto") else None
        task_str = task if task in ("transcribe", "translate") else "transcribe"
        
        segments_gen, info = model.transcribe(
            tmp_path,
            language=lang,
            task=task_str,
            beam_size=5,
            vad_filter=True,          # Skip silent parts
            vad_parameters=dict(min_silence_duration_ms=500),
        )
        
        # Collect segments
        segments = []
        full_text_parts = []
        for i, seg in enumerate(segments_gen):
            text = seg.text.strip()
            segments.append({
                "id": i,
                "start": round(seg.start, 3),
                "end": round(seg.end, 3),
                "text": text,
            })
            full_text_parts.append(text)
        
        return {
            "text": " ".join(full_text_parts),
            "segments": segments,
            "language": info.language,
            "duration": round(info.duration, 2),
        }
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    
    finally:
        os.unlink(tmp_path)


print("✅ FastAPI server configured")

In [ ]:
# ── Cell 5: Start server + expose via ngrok ───────────────────────────────────
from pyngrok import ngrok, conf
import asyncio

PORT = 8000

# ─── Optional: Add your ngrok auth token for longer sessions ─────────────────
# Sign up free at https://dashboard.ngrok.com  →  Your Authtoken
# Without a token: sessions last 2 hours, 1 tunnel at a time
# With a free token: sessions last until Colab disconnects
NGROK_AUTH_TOKEN = ""   # ← Paste your token here, or leave empty

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok auth token set")
else:
    print("ℹ️  Running without ngrok auth token (2-hour limit). Add one above for longer sessions.")

# Start ngrok tunnel
public_url = ngrok.connect(PORT, "http")
url_str = str(public_url).strip("NgrokTunnel: \"'")

print()
print("=" * 60)
print(f"🚀 TurboScript backend is LIVE!")
print()
print(f"   Public URL: {public_url}")
print()
print("📋 COPY the URL above and paste it in:")
print("   TurboScript web app → Settings → Colab URL")
print("=" * 60)
print()
print(f"📡 Local API docs: http://localhost:{PORT}/docs")
print(f"🔍 Health check:   {public_url}/health")
print()
print("⏳ Starting uvicorn server (keep this cell running)...")

# Run uvicorn in background
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())